In [1]:
from data import api

In [ ]:
# get panel data
df = api.get_panel_data()
df

In [ ]:
from models.LGBM.lgbm import prepare_panel_for_lgbm, fit_lgbm_panel, lgbm_diagnostics, lgbm_gof_panel_all_depts, build_lgbm_residuals_geodf, plot_lgbm_spatial_residuals, lgbm_rolling_cv_panel, lgbm_shap_summary

df_feat, X, y, meta_lgbm = prepare_panel_for_lgbm(df, "NUMBER_OF_HOMICIDIO")
model_lgbm, df_feat_sorted, metrics_lgbm = fit_lgbm_panel(df_feat, X, y, meta_lgbm)
print(metrics_lgbm)

gof_global, df_with_resid = lgbm_diagnostics(model_lgbm, df_feat, X, y)

# per-dept GOF
gof_by_dept = lgbm_gof_panel_all_depts(df_with_resid)

# spatial residuals
gdf_resid_lgbm = build_lgbm_residuals_geodf(
    df_with_resid, "data/colombia_departments.geojson"
)
plot_lgbm_spatial_residuals(gdf_resid_lgbm)

shap_values, X_sample, shap_importance = lgbm_shap_summary(
    model_lgbm, df_feat, meta_lgbm, max_samples=2000, show_plots=True
)

cv_lgbm = lgbm_rolling_cv_panel(df_feat, meta_lgbm, n_folds=3, horizon_months=6)
print(cv_lgbm)

# after fitting model_lgbm
gof_global, df_with_resid = lgbm_diagnostics(
    model=model_lgbm,
    df_feat=df_feat,
    X=X,
    y=y,
    lags=24,
)





In [ ]:
from models.LGBM.lgbm import lgbm_shap_summary, lgbm_shap_dependence

# after fitting model_lgbm
shap_vals, X_shap, shap_imp = lgbm_shap_summary(
    model_lgbm,
    df_feat,
    meta_lgbm,
    max_samples=3000,
    show_plots=True,
)

print(shap_imp.head(10))   # top 10 most important features

# detailed dependence for the most important feature
top_feature = shap_imp["feature"].iloc[0]
lgbm_shap_dependence(shap_vals, X_shap, feature=top_feature)
